In [8]:
import pandas as pd

In [9]:
train_df = pd.read_csv('train.csv', sep=';')
test_df  = pd.read_csv('test_no_labels.csv', sep=';')

test_df.columns = ['description', 'sample_name', 'transcription', 'keywords']

print(train_df.shape)
print(test_df.shape)

(2616, 5)
(408, 4)


In [10]:
def clean(df: pd.DataFrame) -> pd.DataFrame:

    df = df.replace(r'^\s*$', None, regex=True)
    df = df.dropna(how='all')

    df = df.apply(lambda col: col.str.lower() if pd.api.types.is_string_dtype(col) else col)

    df['keywords'] = df['keywords'].str.replace('/', ',')
    df['keywords'] = df['keywords'].str.replace(' ', '')


    if 'medical_specialty' in df.columns:    # ONLY training data
        df = df[df['medical_specialty'].notna()]
        
        class_count = df.medical_specialty.value_counts()
        df = pd.merge(df, class_count, on=['medical_specialty'])
        df = df[df['count'] != 1]

        df = df.drop(columns=['count'])

    

    df['keywords']      = df['keywords'].str.replace('.', ' ').str.replace(';', ' ').str.replace(':', ' ').str.replace('-', ' ').str.replace(r'\band\b', '&', regex=True)
    df['transcription'] = df['transcription'].str.replace(',', ' ').str.replace('.', ' ').str.replace(';', ' ').str.replace(':', ' ').str.replace('-', ' ').str.replace(r'\band\b', '&', regex=True)
    df['description']   = df['description'].str.replace(',', ' ').str.replace('.', ' ').str.replace(';', ' ').str.replace(':', ' ').str.replace('-', ' ').str.replace(r'\band\b', '&', regex=True)
    df['sample_name']   = df['sample_name'].str.replace(',', ' ').str.replace('.', ' ').str.replace(';', ' ').str.replace(':', ' ').str.replace('-', ' ').str.replace(r'\band\b', '&', regex=True)

    df['keywords']      = df['keywords'].str.replace(r'\s+', ' ', regex=True).str.strip()
    df['transcription'] = df['transcription'].str.replace(r'\s+', ' ', regex=True).str.strip()
    df['description']   = df['description'].str.replace(r'\s+', ' ', regex=True).str.strip()
    df['sample_name']   = df['sample_name'].str.replace(r'\s+', ' ', regex=True).str.strip()



    return df.drop_duplicates()


In [11]:
train_df = clean(train_df)

train_df.shape

(2613, 5)

In [12]:
test_df = clean(test_df)

test_df.shape

(404, 4)

In [13]:
train_df.to_csv('../data/train.csv', index=False)
test_df.to_csv('../data/test.csv', index=False)